Annotation checker
 - Buttons : "PV_normal_qc", "PV_heater_qc", "PV_pool_qc", "uncertflag_qc", "delete_qc", "resizing_qc"
 - Results : annotation_qc_results_yourname.gpkg

In [ ]:
import geopandas as gpd
import rasterio
from PIL import Image, ImageTk, ImageDraw
import tkinter as tk
from tkinter import ttk
import numpy as np
import os

# File path put your gpkg filepath in 'GPKG_PATH'
GPKG_PATH = "/Users/ilseoplee/cape_town_annotation_checker/2.sample_select_for_qc/sampled_part1_Shawn.gpkg"
IMAGE_FOLDER = "/Users/ilseoplee/cape_town_annotation_checker/1.db_pipeline/download/images"
OUTPUT_PATH = "/Users/ilseoplee/cape_town_annotation_checker/annotation_qc_results_yourname.gpkg"

# Data load
gdf = gpd.read_file(GPKG_PATH)
gdf["fid"] = gdf.index

# QC column
qc_cols = ["PV_normal_qc", "PV_heater_qc", "PV_pool_qc", "uncertflag_qc", "delete_qc", "resizing_qc"]
for col in qc_cols:
    if col not in gdf.columns:
        gdf[col] = 0

class QCChecker:
    def __init__(self, master):
        self.master = master
        self.index = 0
        self.tk_img = None

        self.label = tk.Label(master)
        self.label.pack()

        self.info = tk.Label(master, text="", font=("Arial", 12), justify="left")
        self.info.pack()

        button_frame = ttk.Frame(master)
        button_frame.pack()

        for i, col in enumerate(qc_cols):
            ttk.Button(button_frame, text=col, command=lambda c=col: self.mark(c)).grid(row=0, column=i, padx=5)

        self.next()

    def mark(self, col):
        for qc in qc_cols:
            gdf.at[self.index, qc] = 1 if qc == col else 0
        try:
            gdf.to_file(OUTPUT_PATH, driver="GPKG")
            print(f"Saved after ID {gdf.iloc[self.index].get('id', self.index)}")
        except Exception as e:
            print(f"Save failed: {e}")
        self.index += 1
        self.next()

    def next(self):
        while self.index < len(gdf):
            row = gdf.iloc[self.index]
            image_name = row.get("image_name")
            image_path = os.path.join(IMAGE_FOLDER, image_name + ".tif")

            try:
                with rasterio.open(image_path) as src:
                    geom = row.geometry
                    transform = src.transform
                    centroid = geom.centroid
                    cx, cy = ~transform * (centroid.x, centroid.y)

                    half_w = 300
                    half_h = 300
                    box_crop = (
                        int(cx - half_w),
                        int(cy - half_h),
                        int(cx + half_w),
                        int(cy + half_h)
                    )
                    # clip to image bounds
                    box_crop = (
                        max(0, box_crop[0]),
                        max(0, box_crop[1]),
                        min(src.width, box_crop[2]),
                        min(src.height, box_crop[3])
                    )
                    window = rasterio.windows.Window(
                        col_off=box_crop[0],
                        row_off=box_crop[1],
                        width=box_crop[2] - box_crop[0],
                        height=box_crop[3] - box_crop[1]
                    )
                    data = src.read([1, 2, 3], window=window)
                    win_transform = src.window_transform(window)

                    rgb = np.transpose(data, (1, 2, 0))
                    rgb = np.nan_to_num(rgb)
                    if rgb.dtype != np.uint8:
                        rgb = ((rgb - rgb.min()) / (rgb.ptp() + 1e-6) * 255).astype(np.uint8)

                    img = Image.fromarray(rgb)
                    draw = ImageDraw.Draw(img)

                    if hasattr(geom, "exterior"):
                        coords = list(geom.exterior.coords)
                        pixels = [~win_transform * (x, y) for x, y in coords]
                        pixels = [(int(x), int(y)) for x, y in pixels]
                        if len(pixels) > 2:
                            draw.polygon(pixels, outline="red", width=3)

                    if img.width > 800 or img.height > 800:
                        img.thumbnail((800, 800), Image.LANCZOS)

                    self.tk_img = ImageTk.PhotoImage(img)
                    self.label.configure(image=self.tk_img)
                    self.label.image = self.tk_img

                    self.info.config(
                        text=(
                            f"ID: {row.get('id', 'NA')} | image: {image_name} | annotator: {row.get('annotator', 'NA')}\n"
                            f"PV_normal: {row.get('PV_normal')}, "
                            f"PV_heater: {row.get('PV_heater')}, "
                            f"PV_pool: {row.get('PV_pool')}, "
                            f"uncertflag: {row.get('uncertflag')}"
                        )
                    )
                    return

            except Exception as e:
                print(f"Error loading {image_path}: {e}")

            self.index += 1

        gdf.to_file(OUTPUT_PATH, driver="GPKG")
        print("Save!")
        self.master.quit()

root = tk.Tk()
root.title("Annotation QC Checker")
app = QCChecker(root)
root.mainloop()
